# 数学模块 01｜矩阵求导、Jacobian 与条件数

> **来源**：d2l 2.4.3–2.5（梯度与链式法则）｜Boyd《Convex Optimization》附录 A｜MIT 18.06
> **目标**：把"梯度 / 雅可比 / 曲率"从符号变成直觉，能闭卷推出常用求导公式，并解释归一化为什么能加速收敛
> **前置**：day2（张量 shape 与 autograd）｜day3（梯度下降）
> **学习日期**：09-14 ｜ **验收状态**：⬜ 未做（第 6 节 5 项"必须"全过才算学完）

---

## 0. 五秒回忆卡（闭卷先答）

1. `∂L/∂x` 的形状和什么一致？
2. `∂(xᵀAx)/∂x` 等于什么？A 对称时又是多少？
3. 特征尺度差 1000 倍会让梯度下降变慢，这个现象的数学名字是什么？

---

## 1. 为什么今天学这个

- day2 你建立了"shape 直觉"，day3 你会写 `w -= lr * grad`——但**梯度为什么是这个形状**，还没说清。
- day9 的 BP 四公式里那个"乘 Wᵀ"，本质就是矩阵求导的链式法则（VJP）。补上它，BP 从背诵变成推导。
- day12 你写过"特征尺度差 1000 倍 → 狭长椭圆谷 → 难收敛"。这句话的数学名字就是**条件数**。

一句话：今天学的是"梯度的语法"，它同时服务于反向传播、优化器和归一化三处。

---

## 2. 四个必背概念

### 2.1 布局约定：先声明，再用公式

同一个 `∂y/∂x`，不同教材的形状可能完全相反，原因是**布局约定**不同：

| 约定 | `∂L/∂x`（L 是标量，x ∈ Rⁿ） | `∂y/∂x`（y ∈ Rᵐ, x ∈ Rⁿ） |
| --- | --- | --- |
| 分母布局（结果跟分母走） | (n,) 列向量 | (n, m) |
| 分子布局（结果跟分子走） | (n,) 列向量 | **(m, n)** ← 标准 Jacobian |

**本笔记统一采用**：标量对向量的梯度是列向量（与 x 同形）；雅可比 `J = ∂y/∂x ∈ R^{m×n}`，`J[i,j] = ∂y_i/∂x_j`。

**PyTorch 的约定**：`x.grad` 的形状**永远等于 x 的形状**。拿不准时，用这一条当检查器。

### 2.2 常用求导表（必须闭卷默写）

| 表达式 | 结果 | 备注 |
| --- | --- | --- |
| `∂(aᵀx)/∂x` | `a` | 线性 |
| `∂(xᵀx)/∂x` | `2x` | 范数平方 |
| `∂(xᵀAx)/∂x` | `(A + Aᵀ)x` | A 对称时 → `2Ax` |
| `∂(Ax)/∂x` | `A` | 雅可比，形状 (m, n) |
| `∂(aᵀXb)/∂X` | `a bᵀ` | 结果是矩阵，与 X 同形 |
| `∂‖X‖_F²/∂X` | `2X` | Frobenius 范数 |
| `∂tr(AX)/∂X` | `Aᵀ` | 迹的求导 |
| `∂ log det X / ∂X` | `X^{-T}` | 概率模型里会用到 |

> 记不住就回到**元素法**：写成 Σ 求和，对单个分量求偏导，再拼回矩阵。这是唯一不会出错的兜底方法。

### 2.3 链式法则 = VJP：反向传播的真身

设 `y = f(x)`、L 是最终标量损失。定义上游梯度 `ȳ = ∂L/∂y ∈ Rᵐ`，则

**x̄ = Jᵀ ȳ，其中 J = ∂y/∂x ∈ R^{m×n}**

反向传播每一步做的就是"**乘上局部雅可比的转置**"，这叫 VJP（vector-Jacobian product）。

**和 day9 的 BP2 对上**：`z^(l+1) = W^(l+1) a^(l)`，所以 `∂z^(l+1)/∂a^(l) = W^(l+1)`，于是

```
δ^(l) = (W^(l+1))ᵀ δ^(l+1) ⊙ σ'(z^(l))
```

那个**转置不是规定，是 VJP 的必然结果**——今天要能自己推出来（练习 3）。

### 2.4 Jacobian 与 Hessian

- **Jacobian** `J = ∂y/∂x ∈ R^{m×n}`：一阶信息，形状 =（输出维度, 输入维度）。
- **Hessian** `H = ∂²L/∂x∂xᵀ ∈ R^{n×n}`：二阶信息，对称矩阵，**特征值 = 各主方向的曲率**。

| Hessian 特征值 | 地形 | 优化表现 |
| --- | --- | --- |
| 全 > 0 | 碗底（局部凸） | 可以一路下山 |
| 有正有负 | **鞍点** | 某些方向下坡、某些方向上坡（day10 讲的那个） |
| 差距极大（且都 > 0） | 狭长山谷 | 震荡着慢慢挪，最难优化 |

---

## 3. 条件数：为什么归一化能加速收敛

**定义**：`κ(A) = λmax / λmin = σmax / σmin ≥ 1`。

**对本节的线性回归**：MSE 的 Hessian 是 `H = (2/n) XᵀX`，所以

```
κ = λmax(XᵀX) / λmin(XᵀX)
```

也就是说，**条件数完全由特征的尺度决定**。

**几何画面**：等高线是一族椭圆。κ 接近 1 时椭圆接近圆，梯度方向几乎直指圆心；κ 很大时椭圆被拉成狭长峡谷——沿长轴曲率小、走得慢，沿短轴曲率大、一步就冲过头，于是**一边震荡一边慢慢往下挪**。这正是 day12 里那句"狭长且倾斜的椭圆谷"。

**两个能直接用的结论**

1. **学习率上限**：`η < 2 / λmax`。尺度差越大，λmax 越大，能用的 η 越小 → 收敛越慢。
2. **收敛速度**：对二次型，误差衰减率约为 `((κ−1)/(κ+1))²`。κ = 1000 时几乎不动；κ = 1 时一步到位。

**反推工程实践（这张对照表就是今天最值钱的产出）**

| 手段 | 数学上做了什么 |
| --- | --- |
| 输入标准化 / 白化 | 直接把 κ 压小 |
| BatchNorm / LayerNorm | 每一层再"压扁"一次椭圆 |
| Adam / RMSProp 的逐参数自适应步长 | 相当于**隐式预条件**，抹平各方向的曲率差异 |
| 二阶方法（牛顿法） | 直接用 H⁻¹ 把椭圆"校正"成正圆 |

---


## 4. 手推练习（今天必须动笔，写在下面新增的 markdown cell 里）

1. **元素法**推 `∂(xᵀAx)/∂x = (A + Aᵀ)x`：写出 Σ 形式 → 对单个 `x_k` 求偏导 → 拼回矩阵。
2. 推线性回归 MSE 的 Hessian：`L(w) = (1/(2n))‖Xw − y‖²`，求 `∂²L/∂w∂wᵀ`，并说明什么时候 κ 会很大。
3. 从 VJP 公式推 day9 的 BP2：`δ^(l) = (W^(l+1))ᵀ δ^(l+1) ⊙ σ'(z^(l))`。

---

## 5. 代码实验（补全 TODO 后运行）

三个实验分别对应：条件数怎么算、条件数怎么影响收敛、公式对不对。**先自己写，卡住了再看我的批改。**


In [ ]:
# 实验 1：特征尺度如何决定条件数
import numpy as np

np.random.seed(0)
n = 200
x1 = np.random.randn(n)            # 尺度 ~1
x2 = 1000 * np.random.randn(n)     # 尺度 ~1000
X = np.stack([x1, x2], axis=1)     # (200, 2)

# TODO 1：计算 XᵀX 的特征值，并打印条件数 kappa = λmax / λmin
# 提示：np.linalg.eigvalsh(G) 返回升序特征值
G = X.T @ X
eig = None          # <- 改成你的代码
print("特征值:", eig)
print("kappa :", None)   # <- 改成你的代码


In [ ]:
# 实验 2：同一个学习率，标准化前后的收敛速度对比
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
n = 200
x1 = np.random.randn(n)
x2 = 1000 * np.random.randn(n)
X = np.stack([x1, x2], axis=1)
w_true = np.array([2.0, 0.001])                 # 第二个特征尺度大，对应权重小
y = X @ w_true + 0.1 * np.random.randn(n)

def gd(X, y, lr, steps=200):
    n, d = X.shape
    w = np.zeros(d)
    losses = []
    for _ in range(steps):
        # TODO 2：前向 -> 算 MSE -> 算梯度 -> 更新 w
        # 梯度 = (2/n) * X.T @ (X @ w - y)
        losses.append(None)     # <- 改成真实的 loss
    return losses

# TODO 3：分别用 X 和标准化后的 Xn 跑一次（建议 lr = 1e-6 起步自己调），
#         把两条 loss 曲线画在同一张图上（建议 log 轴），并解释差异
# Xn = (X - X.mean(axis=0)) / X.std(axis=0)


In [ ]:
# 实验 3：用 autograd 验证你手写的求导公式
import torch

torch.manual_seed(0)
x = torch.randn(4, dtype=torch.double, requires_grad=True)
A = torch.randn(4, 4, dtype=torch.double)
a = torch.randn(4, dtype=torch.double)

# TODO 4：先对 f1 = a @ x 求梯度，验证它等于 a
f1 = a @ x
f1.backward()
print("autograd:", x.grad)
print("手推    :", None)      # <- 改成 a

# TODO 5：清零后对 f2 = x @ A @ x 求梯度，验证它等于 (A + A.T) @ x
x.grad = None
f2 = x @ A @ x
f2.backward()
print("autograd:", x.grad)
print("手推    :", None)      # <- 改成 (A + A.T) @ x，并检查两者是否接近


## 6. 验收标准（我要求你达到的程度）

**必须（闭卷，任一项不过就不算学完）**

1. 说清"分母布局 vs 分子布局"的差别，说出 PyTorch 用哪种，并给出那句一句话检查法
2. 默写表格里的 5 个公式：`∂(aᵀx)/∂x`、`∂(xᵀx)/∂x`、`∂(xᵀAx)/∂x`、`∂(Ax)/∂x`、`∂(aᵀXb)/∂X`
3. 用元素法独立推出 `∂(xᵀAx)/∂x = (A + Aᵀ)x`
4. 写出 VJP 公式 `x̄ = Jᵀȳ`，并把 BP2 里的 `Wᵀ` 解释成它的必然结果
5. 说出条件数的定义与几何意义，写出学习率上限 `η < 2/λmax`，并解释"特征尺度差 1000 倍"会发生什么

**加分（做到说明真的通透了）**

6. 分别解释 BatchNorm 和 Adam 是怎么"降低条件数 / 抵消条件数影响"的
7. 用三行 numpy 算出真实数据的 κ，先预测学习率该取多少，再用实验验证预测
8. 解释 Hessian 正定与不定分别为什么对应碗底和鞍点

---

## 7. 自测（3+1）

1. **概念**：为什么 `∂y/∂x` 在不同教材里形状不一样？你怎么快速判断该用哪个？
2. **计算**：`A = [[2, 1], [0, 3]]`，求 `∂(xᵀAx)/∂x`；并指出 A 对称与不对称时的差别。
3. **计算**：Hessian 的特征值是 1000 和 1，写出 η 的上限、给一个安全取值，并估算收敛快慢。
4. **编程**：造一份两特征尺度差 1000 倍的数据，用同一个 η 跑"标准化前 / 标准化后"各一次，把两条 loss 曲线画在一张图上。

---

## 8. 待办

- [ ] 第 4 节三道手推题（写在 notebook 里，不是只在脑子里过）
- [ ] 第 5 节三个代码实验
- [ ] 把 2.2 表格默写一遍后再对照原文
- [ ] 晚上答一组验收题（我按第 6 节的 5 项"必须"出题）
